# Milestone 3 Analysis

Testing the full delivery system: dispensing mechanism, flight controller with the box, and optimized paths. One section per task, each ending in the table its completion criteria ask for.

Every section finds its logs by filename pattern (set in **Configuration**) and skips cleanly if none exist yet, so the notebook runs top to bottom at any stage of the milestone.

## Setup

In [ ]:
import json, glob, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ae483.tools import load_hardware_data, resample_data_drone
from ae483.mytools import sync_data_mocap

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
C_INK, C_MUTED, C_GRID = '#3d3d3a', '#8a8983', '#e4e3df'
plt.rcParams.update({
    'axes.grid': True, 'grid.color': C_GRID, 'grid.linewidth': 0.8,
    'axes.edgecolor': C_MUTED, 'axes.labelcolor': C_INK, 'xtick.color': C_INK, 'ytick.color': C_INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.8, 'legend.frameon': False, 'font.size': 11,
})
pd.set_option('display.precision', 2)

## Configuration

Filename patterns for each task's logs, and the earlier no-box flights used as baselines. Edit these to match how the flight scripts name their files.

In [ ]:
HERE = Path('.')
M2 = Path('../milestone2')

LOGS = {
    'task1': 'task1_release_run*.json',   # ground release-angle tests
    'task2': 'task2_hover_*_run*.json',   # hover with box: empty / partial / full
    'task3': 'task3_path_*_run*.json',    # optimized path with box, no dispensing
    'task4': 'task4_release_run*.json',   # in-flight release
    'task5': 'task5_mission_run*.json',   # full missions
}

# No-box baselines (Nov 6 tasks). Fill in once you know which files they are.
BASELINE_HOVER = []   # e.g. [M2 / 'some_nobox_hover.json']
BASELINE_PATH = []    # e.g. [M2 / 'some_nobox_path.json']

# Completion thresholds from the milestone plan
ANGLE_TOL_DEG = 10.0      # task 1
TARGET_TOL_T3 = 0.20      # task 3: fraction of targets within 20 cm
TARGET_TOL_T5 = 0.05      # task 5: every target within 5 cm
LAND_TOL_T5 = 0.07        # task 5: land within 7 cm of start

def files(task):
    def run_no(p):
        m = re.search(r'run(\d+)', p.name)
        return int(m.group(1)) if m else 0
    return sorted(HERE.glob(LOGS[task]), key=lambda p: (p.name.split('_run')[0], run_no(p)))

## Shared loading and metrics

`load_flight` returns one dict per log: time, setpoint, onboard estimate, and laptop mocap synced into the drone frame (ground truth). Metrics below all use mocap.

In [ ]:
def load_flight(filename):
    filename = Path(filename)
    with open(filename) as f:
        meta = json.load(f)
    raw_drone, raw_mocap, _ = load_hardware_data(str(filename))
    d = resample_data_drone(raw_drone, t_min_offset=0., t_max_offset=0.)
    m = sync_data_mocap(raw_mocap, d['time'], d['ae483log.p_z'], do_transform=True,
                        t_min_offset=0., t_shift_rng=1., t_shift_res=0.05)
    log = lambda n: d[f'ae483log.{n}']
    F = dict(name=filename.name, meta=meta, t=d['time'], raw=d, mocap=m)
    F['p_des'] = np.column_stack([log('p_x_des'), log('p_y_des'), log('p_z_des')])
    F['p_est'] = np.column_stack([log('p_x'), log('p_y'), log('p_z')])
    F['p'] = np.column_stack([m['x'], m['y'], m['z']])
    F['rpy'] = np.column_stack([m['roll'], m['pitch'], m['yaw']])   # mocap attitude, rad
    on = np.flatnonzero(F['p_des'][:, 2] > 0.)
    F['airborne'] = np.zeros_like(F['t'], bool)
    if on.size:
        F['airborne'][on[0]:on[-1] + 1] = True
        F['t_flight'] = F['t'][on[-1]] - F['t'][on[0]]
    return F

def load_all(task):
    fs = files(task)
    if not fs:
        print(f'No {task} logs yet (pattern {LOGS[task]!r}).')
    return [load_flight(f) for f in fs]

def err3(F):
    return np.linalg.norm(F['p'] - F['p_des'], axis=1)

def hold_mask(F):
    """First continuous stretch where the setpoint sits at the hover point."""
    task = F['meta'].get('task5', {}) | F['meta'].get('task', {})
    if 'hold_point' in task:
        hold = np.all(np.isclose(F['p_des'], task['hold_point'], atol=1e-3), axis=1)
    else:
        hold = np.isclose(F['p_des'][:, 2], task.get('z_hover', np.nan), atol=1e-3)
    if not hold.any():
        return hold
    k0 = np.flatnonzero(hold)[0]
    k1 = k0 + np.argmin(hold[k0:]) if not hold[k0:].all() else len(hold)
    out = np.zeros_like(hold); out[k0:k1] = True
    return out

def target_errors(F):
    """Error at the end of the dwell at each target, in visit order.
    A dwell = setpoint xy within 1 mm of a target location."""
    path = F['meta']['path']
    out = []
    for loc in np.asarray(path['path'], float)[1:-1]:   # path starts and ends at start_location
        at = np.all(np.isclose(F['p_des'][:, :2], loc[:2], atol=1e-3), axis=1) & F['airborne']
        if not at.any():
            out.append(np.nan); continue
        k = np.flatnonzero(at)
        k_end = k[np.argmax(np.diff(np.r_[k, k[-1] + 2]) > 1)]   # end of first dwell
        out.append(np.linalg.norm(F['p'][k_end] - F['p_des'][k_end]))
    return np.array(out)

def plot_planned_vs_actual(F, ax=None):
    ax = ax or plt.subplots(figsize=(6, 6))[1]
    a = F['airborne']
    ax.plot(F['p_des'][a, 0], F['p_des'][a, 1], '--', color=C_INK, lw=1.4, label='planned')
    ax.plot(F['p'][a, 0], F['p'][a, 1], color=C_BLUE, label='actual (mocap)')
    locs = np.asarray(F['meta']['path']['path'], float)[1:-1]
    ax.plot(locs[:, 0], locs[:, 1], 'o', color=C_ORANGE, ms=8, mec='white', mew=2, label='targets')
    ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(F['name'], fontsize=11); ax.legend(loc='best')
    return ax

## Task 1: Release angles on the ground

5 tests tilting the drone to the release angles from the Nov 6 Fourth Task.
**Complete when:** 5 tests on video, achieved roll and pitch within 10° of commanded, and exactly one cube released in at least 2 of 5.

Cube counts come from the videos; enter them in `CUBES_T1`. Commanded angles are read from `meta['release']` (`roll_cmd`, `pitch_cmd` in degrees). Achieved angle = median mocap angle over the last second of the tilt.

In [ ]:
CUBES_T1 = {}   # filename -> cubes released, from video, e.g. {'task1_release_run1.json': 1}

rows = []
for F in load_all('task1'):
    r = F['meta'].get('release', {})
    tilt = F['t'] > F['t'][-1] - 1.0   # TODO: replace with the logged tilt window
    roll, pitch = np.degrees(np.nanmedian(F['rpy'][tilt, :2], axis=0))
    rows.append(dict(run=F['name'], roll_cmd=r.get('roll_cmd'), roll=roll,
                     pitch_cmd=r.get('pitch_cmd'), pitch=pitch,
                     cubes=CUBES_T1.get(F['name'])))
t1 = pd.DataFrame(rows)
if len(t1):
    t1['roll_err'] = (t1.roll - t1.roll_cmd).abs()
    t1['pitch_err'] = (t1.pitch - t1.pitch_cmd).abs()
    t1['within_10deg'] = (t1.roll_err <= ANGLE_TOL_DEG) & (t1.pitch_err <= ANGLE_TOL_DEG)
    n_one = (t1.cubes == 1).sum()
    display(t1)
    print(f'Tests: {len(t1)}/5 | all within {ANGLE_TOL_DEG:.0f} deg: {t1.within_10deg.all()} | '
          f'exactly one cube: {n_one}/{len(t1)} (need >= 2) -> '
          f'{"COMPLETE" if len(t1) >= 5 and t1.within_10deg.all() and n_one >= 2 else "not yet"}')

## Task 2: Hover with the box

6 hover flights, 2 each in full, partial, and empty.
**Complete when:** every flight has a log, and a table compares RMS and max position error per configuration against the no-box baseline (Nov 6 First Task).

Error is mocap vs setpoint over the hold.

In [ ]:
def hover_stats(F, label):
    h = hold_mask(F)
    e = err3(F)[h]
    return dict(run=F['name'], configuration=label,
                rms_cm=100 * np.sqrt(np.nanmean(e**2)), max_cm=100 * np.nanmax(e),
                hold_s=h.sum() * np.median(np.diff(F['t'])))

rows = [hover_stats(F, 'no box (baseline)') for F in map(load_flight, BASELINE_HOVER)]
for F in load_all('task2'):
    cfg = (F['meta'].get('task') or F['meta'].get('task5') or {}).get('configuration', '?')
    rows.append(hover_stats(F, cfg))
t2 = pd.DataFrame(rows)
if len(t2):
    display(t2)
    display(t2.groupby('configuration')[['rms_cm', 'max_cm']].agg(['mean', 'max']))

## Task 3: Optimized paths with the box, no dispensing

5–10 flights along trajectory-generator paths.
**Complete when:** at least 5 logs with planned-vs-actual plots, and a table compares terminal error at each target with the no-box results, including the fraction within 20 cm.

In [ ]:
def path_rows(F, label):
    e = target_errors(F)
    return [dict(run=F['name'], set=label, target=i + 1, err_cm=100 * x) for i, x in enumerate(e)]

T3 = load_all('task3')
rows = [r for F in map(load_flight, BASELINE_PATH) for r in path_rows(F, 'no box')]
rows += [r for F in T3 for r in path_rows(F, 'box')]
t3 = pd.DataFrame(rows)
if len(t3):
    display(t3.pivot_table(index='run', columns='target', values='err_cm'))
    summary = t3.groupby('set').err_cm.agg(mean='mean', max='max',
                                           within_20cm=lambda x: np.mean(x <= 100 * TARGET_TOL_T3))
    display(summary)
    print(f'Box logs: {len(T3)} (need >= 5)')

if T3:
    n = len(T3); cols = min(n, 3)
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(5.5 * cols, 5.5 * -(-n // cols)), squeeze=False)
    for F, ax in zip(T3, axes.flat):
        plot_planned_vs_actual(F, ax)
    for ax in axes.flat[n:]:
        ax.remove()
    fig.tight_layout()

## Task 4: Payload release in flight

5 flights releasing at each target.
**Complete when:** at least 5 logs and videos, each records the release attempt at every target with commanded vs actual roll and pitch at release, and at least n−2 of n targets per flight release exactly one cube.

Release events are read from `meta['releases']` (list of dicts with `t`, `target`, `roll_cmd`, `pitch_cmd` in degrees); the flight script needs to log these. Cube counts per target come from video.

In [ ]:
CUBES_T4 = {}   # filename -> list of cubes released at each target, from video

def release_rows(F):
    out = []
    for ev in F['meta'].get('releases', []):
        k = np.argmin(np.abs(F['t'] - ev['t']))
        roll, pitch = np.degrees(F['rpy'][k, :2])
        out.append(dict(run=F['name'], target=ev['target'], roll_cmd=ev['roll_cmd'], roll=roll,
                        pitch_cmd=ev['pitch_cmd'], pitch=pitch))
    return out

T4 = load_all('task4')
t4 = pd.DataFrame([r for F in T4 for r in release_rows(F)])
if len(t4):
    t4['cubes'] = [CUBES_T4.get(r.run, [None] * 99)[r.target - 1] for r in t4.itertuples()]
    display(t4)
    per = t4.groupby('run').agg(n=('target', 'size'), one_cube=('cubes', lambda c: (c == 1).sum()))
    per['pass'] = per.one_cube >= per.n - 2
    display(per)

## Task 5: Full missions

Take off, follow the optimized path, dispense at each target, return to start.
**A mission succeeds if** every target is reached within 5 cm, at least n−2 of n payloads are delivered, and it lands within 7 cm of the start.
**Complete when:** at least 5 missions are logged with trajectory, total flight time, delivery count, and final error.

In [ ]:
DELIVERED_T5 = {}   # filename -> payloads delivered, from video

rows = []
T5 = load_all('task5')
for F in T5:
    e = target_errors(F)
    start = np.asarray(F['meta']['path']['start_location'], float)
    p_land = F['p'][np.flatnonzero(F['airborne'])[-1]]
    land_err = np.linalg.norm(p_land[:2] - start[:2])
    n = len(e); got = DELIVERED_T5.get(F['name'])
    ok = (np.all(e <= TARGET_TOL_T5) and got is not None and got >= n - 2 and land_err <= LAND_TOL_T5)
    rows.append(dict(run=F['name'], flight_time_s=F.get('t_flight'), targets=n,
                     max_target_err_cm=100 * np.nanmax(e), delivered=got,
                     land_err_cm=100 * land_err, success=ok))
t5 = pd.DataFrame(rows)
if len(t5):
    display(t5)
    print(f'Missions logged: {len(t5)} (need >= 5) | successful: {t5.success.sum()}')
    n = len(T5); cols = min(n, 3)
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(5.5 * cols, 5.5 * -(-n // cols)), squeeze=False)
    for F, ax in zip(T5, axes.flat):
        plot_planned_vs_actual(F, ax)
    for ax in axes.flat[n:]:
        ax.remove()
    fig.tight_layout()

## Summary

In [ ]:
status = {
    'Task 1': len(t1) >= 5 and bool(t1.get('within_10deg', pd.Series([False])).all()) and (t1.get('cubes') == 1).sum() >= 2 if len(t1) else False,
    'Task 2': len(t2[t2.configuration != 'no box (baseline)']) >= 6 if len(t2) else False,
    'Task 3': len(T3) >= 5,
    'Task 4': len(T4) >= 5 and bool(per['pass'].all()) if len(t4) else False,
    'Task 5': len(t5) >= 5,
}
pd.Series(status, name='complete').to_frame()